# Game Engine

In this notebook, we're going to complete the objective for week three: **building the engine**. The engine will be based on our flagship version of an LLM—Gemini 3 Flash—which was prompted using few-shot, chain of thought (CoT), and versioning. Additionally, we'll be supporting input and output (I/O) from the user. This means:

* Piping the relevant information from our latest few-shot file
* Creating an architecture that incorporates our setlist of skits
* Defining the logic behind the branching (if any)
* Toying with other prompt practices to make NPC interactions better

In [2]:
# Add your dependencies (may need more/less; TBD)
import textwrap
import random
import re
import time
import csv
import io
from google import genai
from google.genai import types
from google.colab import userdata

# Fetch API key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key = GEMINI_API_KEY)

In [3]:
#upload comedy data
from google.colab import files
uploaded = files.upload()

Saving DS 340 Final Project Data_test.csv to DS 340 Final Project Data_test.csv


In [4]:
#csv read, extract joke and rating
def fewshot_data_split(uploaded):
  file_name = list(uploaded.keys())[0]
  file_content = uploaded[file_name]
  csv_text = file_content.decode('latin-1')
  csv_reader = csv.reader(io.StringIO(csv_text, newline = ''))
  next(csv_reader)

  testing_sets = []

  for row in csv_reader:
    if not row:
      continue
    try:
      row_id = int(row[0])
      joke = row[1]
      rating = str(row[2])

      # testing sets (15 of each to avoid class imbalance)
      if (1 <= row_id <= 15) or (51 <= row_id <= 65):
        testing_sets.append({
            "id": row_id,
            "joke": joke,
            "rating": rating
            })

    except (ValueError, IndexError):
      continue

  training_sets = [
      # Hard-coded three examples of class 0 & reasons for their classification
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: I used to date Hispanic guys, but now I prefer consensual.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is a racist rape joke mocking Hispanic men, making it both overly crude and insensitive. \nScore: 0")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: You can get a lot of power out of one inch.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is low and almost too easy of a joke to make, so it's unoriginal and not funny. \nScore: 0")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: No one notices the Taylor oil spill because it's a disaster taking place over a long period of time, like Derrick Rose's career.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This comes off as antagonistic since it throws shade at a person, Derrick Rose (whom many consider to be a great but flawed NBA player), and is too unpopular of an opinion, thus not constituting a good joke. \nScore: 0")]),

      # Hard-coded three examples of class 1 & reasons for their classification
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Please, don't call me sir. Call me ma'am.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because it is witty and self-deprecating, given that the user is indeed a man. \nScore: 1")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Hey, I am Conan O'Brien and I'm honored to be the last human host of the Academy Awards. Yes! Yeah! Next year, it's going to be a Waymo in a tux.")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is very funny as it's clever and uses observational comedy to comment on the advent of AI in self-driving cars, like Waymo, whilst keeping it wholesome. \nScore: 1")]),
      types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: How did us teaching Diana to drive turn into I'm a male prostitute, you're going to put me out, and you're going to come back in an hour, and you want your trap full?")]),
      types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because, in spite of its unorthodox format (as it doesn't read like a classic joke or pun), it calls out the absurdity of the situation in a unique and unpredictable way. \nScore: 1")])
  ]

  return training_sets, testing_sets

In [19]:
# Skits
skits = {
    "skit1": {
        "description": "The user is a college exchange student who wants to sound like a native,"
            "so they take a crash course in a foreign language—with special emphasis on learning slang—from an instructor and dialect coach."
            "It is provided that the LLM gives translations for the other tongue in parentheses (akin to subtitles) so that the user may respond in English. ",
              "stages":
               [
                  {
                  "step" : 1,
                  "goal" : "User learns how to say their name, greetings, phrases, questions, & body parts",
                  "max_exchanges" : 1
                  },
                  {
                  "step" : 2,
                  "goal" : "User’s newly learned slang skill accidentally stops a world-ending crisis",
                  "max_exchanges" : 1
                  }
              ]
            },
    "skit2": {
        "description": "The user crashes a club meeting they have never been to before. Is it for freebies?"
            "Is it for…um, kicks and giggles? Maybe getting a guy or girl’s number or something? ",
            "stages":
             [
                {
                  "step" : 1,
                  "goal" : "User sees boxes of Dunkin’ Donuts and Baskin-Robbins on the table, but doesn’t know if they should eat it or not since no one says anything",
                  "max_exchanges" : 1
                },
                {
                  "step" : 2,
                  "goal" : "No donuts remain on the table",
                  "max_exchanges" : 1
                }
             ]
            },
    "skit3": {
        "description": "The user takes a part-time job at a convenience store, cafe, or cram school to make some extra money on the side."
            "And because they want that Employee of the Month (EOM) reward. For a genuine, authentic experience, it sure is a hell of a lot of work. ",
              "stages":
               [
                  {
                  "step" : 1,
                  "goal" : "User is tasked with:"
                  "1. Welcoming customers, helping them find goods, etc. (convenience store)"
                  "Being a cashier, receiving calls for orders, serving customers, etc. (cafe)",
                  "max_exchanges" : 1
                  },
                  {
                  "step" : 2,
                  "goal" : "User misses orders and messes up the procedure; despite all these, they still earns the EOM (Employee of the Month) award",
                  "max_exchanges" : 1
                  }
              ]
            },
    "skit4": {
        "description": "The user surprises their older brother who is in the middle of a date with his new girlfriend."
            "Like every younger sibling, they proceed to third-wheel for the rest of the day. Poor child… ",
              "stages":
               [
                {
                  "step" : 1,
                  "goal" : "User is tasked with: "
                  "User accompanies their brother and his girlfriend to a romantic dinner. On the dinner table, the user decides to do something to smoothen the tension…",
                  "max_exchanges" : 1
              },
               {
                  "step" : 2,
                  "goal" : "The user out-impresses the girlfriend and earns her secret validation",
                  "max_exchanges" : 1
              }
              ]
            },
    "skit5": {
        "description": "The user is about to finish their exchange program, so they try to cook their homestay dinner as a thank you."
            "Time for a solo visit to the grocery store with a recipe they found on YouTube.",
              "stages":
               [
                {
                  "step" : 1,
                  "goal" : "User starts out at a supermarket where they interact with people to find ingredients, then goes back home to the kitchen they never used",
                  "max_exchanges" : 1
                },
                {
                  "step" : 2,
                  "goal" : "User destroys parts of the kitchen, but surprisingly results in Michelin-star level food.",
                  "max_exchanges" : 1
                }
              ]
            }
}

In [25]:
evaluator_instruction = (
    "You are a professional comedy writer. Evaluate whether the text is funny or not based on this binary rubric:\n"
    "0 (not funny) if there's no comedic intent or feels forced.\n"
    "1 (funny) if it's unpredictable yet clever, intuitive in its wordplay, or isn't overty offensive. Be more lenient in the grading of 1 and forgive the user for unoriginal input.\n"
    "First, write one brief sentence explaining your reasoning. Then, on a new line, output the final score in this exact format: 'Score: [0/1]'."
)
training_sets, testing_sets = fewshot_data_split(uploaded)

def get_humor_score(player_text):
    turn_input = types.Content(role = 'user', parts = [types.Part.from_text(text = f"Text: {player_text}")])
    response = client.models.generate_content(
    model = "gemini-3-flash-preview",
    contents = training_sets + [turn_input],
    config = types.GenerateContentConfig(
        system_instruction = evaluator_instruction,
        temperature = 0
      )
    )
    match = re.search(r'Score:\s*([0-1])', response.text)
    return int(match.group(1)) if match else 0

In [24]:
# Create the game / dungeon master whose job is to add to & connect the skits
def skitzo(skit_skeleton):
    prompt = f"""You are the (dungeon) master for an interactive text-based game in the style of 'Choice of Games.' Your job is to take a basic outline of a skit and expand it into an immersive RPG scene using the following pointers:

    1. Narrate in the second person (e.g. 'You pull up to...' or 'Your heart races.')
    2. Write one to four paragraphs totalling no more than a dozen sentences. Let the complexity of the outline dictate the exact length, just like a real novel
    3. Ground the player in the scene by describing the setting (i.e. one or more of the five senses—sight, smell, touch, taste, hearing)
    4. Include at least one piece of dialogue from an NPC interwoven into the scene
    5. End the final paragraph with an NPC interacting directly to the player. Leave the narrative hanging so that the player must respond.
    6. Remember that the user is an exchange student studying in a foreign country, stick all skits based on this setting.

    Here's the outline to expand upon: {skit_skeleton}"""

    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            temperature = 0.7
        )
    )

    return response.text

def npc(scenario, player_text, score):
    reaction_type = "positive, laughing, and rewarding" if score == 1 else "awkward, offended, or confused"
    prompt = f"""
    Context: {scenario}
    Player's response: {player_text}

    The player's response was evaluated for humor and got a score of {score}/1. Write a short, {reaction_type} response from the NPC reacting to the player."""

    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            temperature = 0.7
        )
    )

    return response.text

In [22]:
# Run the game
def run_game():
    print("Insert cold open here...\n")
    total_score = 0
    total_exchanges = 0 # Track for final grading

    # Loop through every skit
    for skit_id, skit_data in skits.items():
        print("=" * 100)
        print(f"PLAYING SKIT: {skit_id.upper()}".center(100))
        print("=" * 100 + "\n")

        # Keep a running history so that the LLM remembers what happened in Step 1 when playing Step 2
        running_story = f"Setting: {skit_data['description']}"

        # Loop through every stage within the current skit
        for stage in skit_data["stages"]:
            step_num = stage["step"]
            goal = stage["goal"]
            max_io = stage["max_exchanges"]

            print(f"\n{'-'*40} STAGE {step_num} {'-'*40}")

            # Generate the next part of the story based on the current goal using skitzo
            stage_prompt = f"Previous context: {running_story}\nNow, move the story forward. The new goal is: {goal}."
            scene_setup = skitzo(stage_prompt)

            wrapped_scene = textwrap.fill(scene_setup, width=100)
            print(wrapped_scene)

            # Add narration
            running_story += f"\nNarration: {scene_setup}"

            # Loop for the exact number of max_exchanges (I/O) defined in the stage
            for exchange_num in range(max_io):
                player_action = input("\nWhat do you do/say? > ")

                # Evaluate Humor
                score = get_humor_score(player_action)
                total_score += score
                total_exchanges += 1

                # Generate NPC reaction
                reaction = npc(running_story, player_action, score)
                wrapped_reaction = textwrap.fill(reaction, width=100)

                print(f"\n[Evaluator Score: {score}/1]")
                print(f"NPC: {wrapped_reaction}\n")

                # Add the player's action and NPC's reaction to the history so that the LLM does not forget
                running_story += f"\nPlayer did/said: {player_action}\nNPC reacted: {reaction}"
                time.sleep(1)

    # Final branching endings based on the total number of exchanges
    print("=" * 100)
    print("GAME OVER".center(100))
    print("=" * 100 + "\n")

    print(f"Final Score: {total_score} out of {total_exchanges}\n")

    # Grading
    if total_score >= (total_exchanges * 0.8):
        ending_type = "GOOD ENDING: The student is awarded and life quality improves drastically."
        status = "Success/Hero"
    elif total_score >= (total_exchanges * 0.4):
        ending_type = "NEUTRAL ENDING: The student continues daily life as normal."
        status = "Average/Surviving"
    else:
        ending_type = "BAD ENDING: The student is jailed or expelled from the country."
        status = "Failure/Criminal"

    # 2. Build a prompt for the final story
    # We use 'running_story' which contains the log of all 5 skits
    final_prompt = f"""
    You are the narrator of an interactive comedy game about an exchange student.
    The game is over. Here is the summary of the player's journey:

    {running_story}

    FINAL RESULT: {ending_type}

    Write a cinematic, funny closing narration (2-3 paragraphs) that concludes the student's story
    based on the result above. Reference specific events from the story history to make it feel
    cohesive.
    """

    final_narrative_response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=final_prompt,
        config=types.GenerateContentConfig(temperature=0.7)
    )

    final_text = final_narrative_response.text

    # Output Display
    print("=" * 100)
    print("THE CONCLUSION".center(100))
    print("=" * 100 + "\n")

    print(f"Final Score: {total_score} / {total_exchanges} | Status: {status}\n")

    wrapped_final = textwrap.fill(final_text, width=100)
    print(wrapped_final)
    print("\n" + "=" * 100)

In [27]:
run_game()

Insert cold open here...

                                        PLAYING SKIT: SKIT1                                         


---------------------------------------- STAGE 1 ----------------------------------------
The scent of strong, bitter espresso and old paper fills the cramped office where Marco, your
dialect coach, leans back in his creaking wooden chair. Sunlight filters through the dusty blinds,
illuminating a whiteboard covered in messy scribbles of local street talk and crossed-out textbook
phrases. You’ve spent the morning unlearning the stiff, formal greetings you memorized back home,
and now the real work of sounding like a local begins.  "Forget the polite introductions they taught
you at the airport," Marco says with a smirk, tapping a rhythmic beat on his desk with a pencil. He
points to his chest, then his head, and finally to you, his eyes gleaming with academic mischief.
"To fit in at the late-night cafes or the crowded street markets, you need to sound like you

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}